# Data Preprocessing

In [42]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MultiLabelBinarizer, OrdinalEncoder, StandardScaler, LabelEncoder

df = pd.read_csv("raw_dataset.csv")

### 1. Removing illogical data

#### a. Filter out impossible matches

In [43]:
df = df[df['mutual_matches'] <= df['likes_received']]

#### b. Filter out zero usage time but sent messages

In [44]:
df = df[~((df['app_usage_time_min'] == 0) & (df['message_sent_count'] > 0))]

### 2. Drop redundant column

In [45]:
cols_to_drop = ['app_usage_time_labels', 'swipe_right_label']
df = df.drop(columns=cols_to_drop, errors='ignore')

#### 3. Cyclical Encoding for Time

#### a. Transform the 24-hour clock into sine and cosine components

In [46]:
df['last_active_hour_sin'] = np.sin(2 * np.pi * df['last_active_hour'] / 24)
df['last_active_hour_cos'] = np.cos(2 * np.pi * df['last_active_hour'] / 24)

#### b. Drop the raw hour column so the model doesn't use it linearly

In [47]:
df = df.drop(columns=['last_active_hour']) 

### 4. Encoding the interest_tags

#### a. Fill NaNs, convert to string, and split by the comma delimiter

In [48]:
df['interest_tags'] = df['interest_tags'].fillna('').astype(str).str.split(', ')

#### b. Apply MultiLabelBinarizer

In [49]:
mlb = MultiLabelBinarizer()
tags_encoded = mlb.fit_transform(df['interest_tags'])
tags_df = pd.DataFrame(tags_encoded, columns=mlb.classes_, index=df.index)

#### c. Keep only the top 50 most frequent tags

In [50]:
top_50_tags = tags_df.sum().nlargest(50).index
tags_df_top = tags_df[top_50_tags].copy()

#### d. Group all remaining tags into an 'Other_Tags' column

In [51]:
tags_df_top['Other_Tags'] = tags_df.drop(columns=top_50_tags).sum(axis=1).clip(upper=1)

#### e. Concatenate back to the main dataframe and drop the original text column

In [52]:
df = pd.concat([df.drop(columns=['interest_tags']), tags_df_top], axis=1)

### 5. Encode Other Categorical Variables

In [53]:
le = LabelEncoder()
df['match_outcome'] = le.fit_transform(df['match_outcome'])

education_order = ['High School', 'Bachelor’s', 'Master’s', 'PhD'] 
income_order = ['Low', 'Medium', 'High']

ord_enc_edu = OrdinalEncoder(categories=[education_order], handle_unknown='use_encoded_value', unknown_value=-1)
ord_enc_inc = OrdinalEncoder(categories=[income_order], handle_unknown='use_encoded_value', unknown_value=-1)

df['education_level'] = ord_enc_edu.fit_transform(df[['education_level']])
df['income_bracket'] = ord_enc_inc.fit_transform(df[['income_bracket']])

nominal_cols = ['gender', 'sexual_orientation', 'location_type', 'swipe_time_of_day']
df = pd.get_dummies(df, columns=nominal_cols, drop_first=True)

### 6. Scale the Numerical Features

In [54]:
num_cols = [
    'likes_received', 'mutual_matches', 'bio_length', 'message_sent_count', 
    'app_usage_time_min', 'profile_pics_count', 'swipe_right_ratio', 'emoji_usage_rate'
]
scaler = StandardScaler()
df[num_cols] = scaler.fit_transform(df[num_cols])

## 7. Save the dataset

In [56]:
df.to_csv('cleaned_dataset.csv', index=False)